In [15]:
import uproot
import pandas as pd
import numpy as np
import awkward as ak
import os
import json
import ROOT

In [16]:
os.environ['ttH_yy_DIR'] = '/eos/user/e/elmazzeo/ttH@FCC-hh/results/2025-05-05/'

In [17]:
basedir = os.path.join(os.environ.get('ttH_yy_DIR'), 'final')

process = {
    'ttZee' : { 
        'sample_list' : ['mgp8_pp_ttz01j_5f_84TeV_zee'], 
               'label' : r'$ttZ$ $\rightarrow$ ee', 'kfactor' : 1.},
    'WZjets' : { 
        'sample_list' : ['mgp8_pp_WZjj_HF_5f_84TeV_zeewlep'], 
               'label' : 'WZ+jets', 'kfactor' : 1.},
    'ZZjets' : { 'sample_list' : ['mgp8_pp_ZZjj_HF_5f_84TeV_zzlep'], 
               'label' : 'ZZ+jets', 'kfactor' : 1.},
    'tZj' : { 'sample_list' : ['mgp8_pp_tZj_5f_84TeV_zeewlep'], 
               'label' : 'tZq', 'kfactor' : 1.},
    'tWZj' : { 'sample_list' : ['mgp8_pp_tWZj_5f_84TeV_zee'], 
               'label' : 'tWZ+jets', 'kfactor' : 1.}
}

selection = {
           "nocuts" : "All events", # all events
           "electrons": "$\geq$ 2 $e$",
           "electrons_charge" :  "2 opposite charge $e$",
           "electrons_mee_window": "$Z\to ee$ cand.",
           "preselection": "$Z\to ee$ cand. + $\geq$ 2 $b$-jets",
           "trilep_channel": "Trilepton channel",
           "tetralep_channel" : "Tetralep. channel",
           "lep_channel" : "Combined"
}

variables = ['weight']

process_infos = ["/eos/experiment/fcc/hh/utils/FCCDicts/FCChh_procDict_fcc_v07_II.json"]

lumi = 3e7 # 3 * 10^7 pb-1 = 3 ab-1

In [18]:
process_dicts = []
for inputname in process_infos :
    with open(inputname, 'r') as f :
        process_dicts.append(json.load(f))

In [19]:
df = {}
sow = {}

In [20]:
for p in process.keys() :
    print(p)
    if p in list(df.keys()) :
        pass
    else :
        df[p] = {}
    for s in selection.keys() :
        print("\t"+s)
        if s in list(df[p].keys()) :
            continue
        df1 = []
        sow[p] = []
        for sample in process[p]['sample_list'] :
            print("\t\t"+sample)
            inputfile = os.path.join(basedir, sample+"_"+s+".root")
            # get sum of weights
            f = ROOT.TFile.Open(inputfile)
            sow[p].append(f.Get("SumOfWeights").GetVal())
            f.Close()
            # get sample dict
            for d in process_dicts :
                if sample in list(d.keys()) :
                    process_dict = d.copy()
                    break
            # get sample
            with uproot.open(inputfile) as f :
                df1.append(ak.to_dataframe(f['events'].arrays(expressions=variables, library='ak')))
                df1[-1]["weight"] = df1[-1]["weight"]/sow[p][-1]*process_dict[sample]['crossSection']*process_dict[sample]['kfactor']*process_dict[sample]['matchingEfficiency']
        df[p][s] = pd.concat(df1, copy=True, ignore_index=True)
        df[p][s]['weight'] = df[p][s]['weight']*process[p]['kfactor']

ttZee
	nocuts
		mgp8_pp_ttz01j_5f_84TeV_zee
	electrons
		mgp8_pp_ttz01j_5f_84TeV_zee
	electrons_charge
		mgp8_pp_ttz01j_5f_84TeV_zee
	electrons_mee_window
		mgp8_pp_ttz01j_5f_84TeV_zee
	preselection
		mgp8_pp_ttz01j_5f_84TeV_zee
	trilep_channel
		mgp8_pp_ttz01j_5f_84TeV_zee
	tetralep_channel
		mgp8_pp_ttz01j_5f_84TeV_zee
	lep_channel
		mgp8_pp_ttz01j_5f_84TeV_zee
WZjets
	nocuts
		mgp8_pp_WZjj_HF_5f_84TeV_zeewlep
	electrons
		mgp8_pp_WZjj_HF_5f_84TeV_zeewlep
	electrons_charge
		mgp8_pp_WZjj_HF_5f_84TeV_zeewlep
	electrons_mee_window
		mgp8_pp_WZjj_HF_5f_84TeV_zeewlep
	preselection
		mgp8_pp_WZjj_HF_5f_84TeV_zeewlep
	trilep_channel
		mgp8_pp_WZjj_HF_5f_84TeV_zeewlep
	tetralep_channel
		mgp8_pp_WZjj_HF_5f_84TeV_zeewlep
	lep_channel
		mgp8_pp_WZjj_HF_5f_84TeV_zeewlep
ZZjets
	nocuts
		mgp8_pp_ZZjj_HF_5f_84TeV_zzlep
	electrons
		mgp8_pp_ZZjj_HF_5f_84TeV_zzlep
	electrons_charge
		mgp8_pp_ZZjj_HF_5f_84TeV_zzlep
	electrons_mee_window
		mgp8_pp_ZZjj_HF_5f_84TeV_zzlep
	preselection
		mgp8_pp_ZZjj_

In [21]:
df[p][s].head()

,weight
0,5.311462e-07
1,5.311462e-07
2,5.311462e-07
3,5.311462e-07
4,5.311462e-07


# Entries

In [22]:
my_entries = {
    "Selection" : []
}

In [23]:
for p in process.keys() :
    my_entries[process[p]['label']] = []

In [24]:
for s in selection :
    my_entries["Selection"].append(selection[s])
    for p in process.keys() :
        my_entries[process[p]['label']].append(len(df[p][s]))

In [25]:
my_entries = pd.DataFrame(my_entries)
my_entries = my_entries.set_index('Selection')

In [26]:
my_entries

,$ttZ$ $\rightarrow$ ee,WZ+jets,ZZ+jets,tZq,tWZ+jets
Selection,,,,,
All events,3068239,4990000,4990000,5000000,4990000
$\geq$ 2 $e$,1412396,2273235,1418393,2471367,2349593
2 opposite charge $e$,1324846,2066643,1339507,2239941,2202724
$Z\to ee$ cand.,1224385,1857044,1248407,2003978,2033813
$Z\to ee$ cand. + $\geq$ 2 $b$-jets,728913,143580,189993,537046,1089815
Trilepton channel,173089,33322,46535,177844,260135
Tetralep. channel,18749,24,61811,109,28366
Combined,191838,33346,108346,177953,288501


# Yields @ 30 ab-1

In [27]:
my_yields = {
    "Selection" : []
}

In [28]:
for p in process.keys() :
    my_yields[process[p]['label']] = []

In [29]:
for s in selection :
    my_yields["Selection"].append(selection[s])
    for p in process.keys() :
        my_yields[process[p]['label']].append(df[p][s]["weight"].sum()*lumi)

In [30]:
my_yields = pd.DataFrame(my_yields)
my_yields = my_yields.set_index('Selection')

In [31]:
my_yields

,$ttZ$ $\rightarrow$ ee,WZ+jets,ZZ+jets,tZq,tWZ+jets
Selection,,,,,
All events,3.210753e+07,1.355999e+06,1.796763e+06,1.020158e+07,7.963446e+07
$\geq$ 2 $e$,1.478142e+07,6.180139e+05,5.107910e+05,5.043944e+06,3.749918e+07
2 opposite charge $e$,1.386515e+07,5.618434e+05,4.823823e+05,4.571643e+06,3.515512e+07
$Z\to ee$ cand.,1.281375e+07,5.048571e+05,4.495743e+05,4.090050e+06,3.245936e+07
$Z\to ee$ cand. + $\geq$ 2 $b$-jets,7.628590e+06,3.903668e+04,6.842047e+04,1.096075e+06,1.739371e+07
Trilepton channel,1.811469e+06,9.060426e+03,1.675797e+04,3.629748e+05,4.151862e+06
Tetralep. channel,1.961836e+05,6.520742e+00,2.225942e+04,2.226452e+02,4.527399e+05
Combined,2.007653e+06,9.066947e+03,3.901739e+04,3.631974e+05,4.604602e+06


In [82]:
30e6*1.07026/1e7

3.21078

# Efficiency

In [32]:
my_eff = {
    "Selection" : []
}

In [33]:
for p in process.keys() :
    my_eff[process[p]['label']] = []

In [34]:
for s in selection :
    my_eff["Selection"].append(selection[s])
    for p in process.keys() :
        my_eff[process[p]['label']].append(df[p][s]["weight"].sum()/df[p]["nocuts"]["weight"].sum())

In [35]:
my_eff = pd.DataFrame(my_eff)
my_eff = my_eff.set_index('Selection')

In [36]:
my_eff

,$ttZ$ $\rightarrow$ ee,WZ+jets,ZZ+jets,tZq,tWZ+jets
Selection,,,,,
All events,1.000000,1.000000,1.000000,1.000000,1.000000
$\geq$ 2 $e$,0.460373,0.455763,0.284284,0.494428,0.470891
2 opposite charge $e$,0.431835,0.414339,0.268473,0.448131,0.441456
$Z\to ee$ cand.,0.399089,0.372314,0.250213,0.400923,0.407604
$Z\to ee$ cand. + $\geq$ 2 $b$-jets,0.237595,0.028788,0.038080,0.107442,0.218419
Trilepton channel,0.056419,0.006682,0.009327,0.035580,0.052136
Tetralep. channel,0.006110,0.000005,0.012389,0.000022,0.005685
Combined,0.062529,0.006687,0.021715,0.035602,0.057822


# Signal / Background ratio in trilepton, tetralepton and combined region

In [37]:
my_selection = ['trilep_channel', 'tetralep_channel', 'lep_channel']
my_sig = 'ttZee'
my_sig_label = process[my_sig]['label']
my_bkg = ['WZjets', 'ZZjets', 'tZj', 'tWZj']

In [38]:
my_ratio = {
    "Selection" : []
}

In [39]:
for p in my_bkg :
    my_ratio[my_sig_label + " / " + process[p]['label']] = []

In [40]:
for s in my_selection :
    my_ratio["Selection"].append(selection[s])
    for p in my_bkg :
        my_sig_yield = df[my_sig][s]["weight"].sum()*lumi
        my_bkg_yield = df[p][s]["weight"].sum()*lumi
        my_ratio[my_sig_label + " / " + process[p]['label']].append(my_sig_yield/my_bkg_yield)

In [41]:
my_ratio = pd.DataFrame(my_ratio)
my_ratio = my_ratio.set_index('Selection')

In [42]:
my_ratio

,$ttZ$ $\rightarrow$ ee / WZ+jets,$ttZ$ $\rightarrow$ ee / ZZ+jets,$ttZ$ $\rightarrow$ ee / tZq,$ttZ$ $\rightarrow$ ee / tWZ+jets
Selection,,,,
Trilepton channel,199.932029,108.096018,4.990621,0.436303
Tetralep. channel,30086.087815,8.813508,881.148882,0.433325
Combined,221.425472,51.455338,5.527718,0.436010
